# 08 — DOI and document-type extraction (rules, no LLM)

Run from the `notebooks/` folder. Needs `output/inventory/documents.csv`; optional `output/extraction/gold_metadata.csv` for a quick check (the full evaluation is notebook 09).

**Why a separate, rule-based DOI step?** A DOI is an exact string. A small LLM often returns nothing or invents one, while a harvesting step with scoring can read every place where a publisher prints it and *show its evidence*. In your 25 dev documents, notebook 07 found a DOI in only 5.

**Where this notebook looks for DOIs**

| source | what is read | typical for |
|---|---|---|
| `metadata` | PDF info fields and XMP metadata | publisher PDFs |
| `link` | clickable `doi.org` links on pages 1–3 (the link target is often *not* in the page text) | journal articles |
| `text`, zones `HEADER` / `FOOTER` / `MARGIN_*` / `SIDEBAR_*` | the top and bottom 12% of every page, the left and right edges, and vertical (rotated) side text | running headers/footers of journals, copyright blocks of conference papers, side banners |
| `text`, zone `BODY` | the full text of pages 1–6 (citation line, colophon, "DOI:" label) | reports, theses, first-page citation lines |
| `text`, zone `REFERENCES` | text after a "References / Literatuur" heading | **cited** works, so it counts against a DOI |

**How the right DOI is chosen.** Every distinct DOI gets a score (see section 3). Evidence of the document's *own* DOI adds points: it is in the metadata, behind a link, in a page edge, repeated on several pages, or labelled `doi`. Evidence that it is a *cited* DOI subtracts points. Below `MIN_SCORE` nothing is returned, because for many documents (most Dutch reports) there is no DOI at all and a wrong DOI is worse than none.

The notebook also guesses the **document type** from structural cues (journal / conference / report / thesis / policy brief), because your three structures differ and the LLM labelled 15 of 25 documents `conference_paper`.

## 1. Settings

In [4]:
from pathlib import Path
from collections import defaultdict
from urllib.parse import unquote
import re, unicodedata
import pandas as pd
import pymupdf
from IPython.display import display

PDF_DIR = Path('../pdfs').resolve()
INVENTORY_DIR = Path('../output/inventory').resolve()
OUT_DIR = Path('../output/extraction').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

FULL_SCAN_PAGES = 6     # these pages are read completely; later pages only their edges
LINK_PAGES = 3          # clickable links are read on these pages
EDGE = 0.12             # width of the header / footer / side bands, as a fraction of the page
MIN_SCORE = 3           # DOI candidates scoring below this are rejected

read_csv = lambda path: pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
documents = read_csv(INVENTORY_DIR / 'documents.csv')
todo = documents[documents.text_status.isin(['extractable', 'partial'])]
print(f'{len(todo)} documents with text (the {(documents.text_status == "no_text").sum()} without text need OCR first)')

92 documents with text (the 1 without text need OCR first)


## 2. Finding DOIs in text

In [5]:
DOI_RE = re.compile(r'10\.\d{4,9}/[^\s"<>]+', re.I)
REF_HEADING = re.compile(r'^\W*(?:\d+\.?\s*)?(references?|bibliography|literatuur(?:lijst)?|referenties|bronnen(?:lijst)?)\W*$', re.I)
EDGE_ZONES = {'HEADER', 'FOOTER', 'MARGIN_LEFT', 'MARGIN_RIGHT', 'SIDEBAR_LEFT', 'SIDEBAR_RIGHT'}

def clean(text):
    text = unicodedata.normalize('NFKC', text)
    return re.sub(r'[\x00-\x08\x0b-\x1f\ue000-\uf8ff\ufffd]', '', text).strip()

def unwrap(text):
    """Repair DOIs that are broken over two lines or separated from their label."""
    text = re.sub(r'(?i)(doi\.org/|doi\s*:?)[ \t]+(?=10\.\d)', r'\1', text)
    return re.sub(r'(10\.\d{4,9}/\S*[-./_])[ \t]*\n[ \t]*(?=[0-9a-z])', r'\1', text)

def trim_doi(doi):
    doi = doi.rstrip(".,;:]}'\"")
    while doi.endswith(')') and doi.count(')') > doi.count('('):
        doi = doi[:-1].rstrip(".,;:")
    return doi.lower()

def doi_matches(text):
    """Every DOI in the text as (doi, marked). marked = the word 'doi' stands right before it."""
    text = unwrap(text)
    return [(trim_doi(m.group(0)), 'doi' in text[max(0, m.start() - 12):m.start()].lower())
            for m in DOI_RE.finditer(text)]

# quick self-test
assert doi_matches('Available at https://doi.org/10.1016/j.algal.2024.\n103538.')[0] == ('10.1016/j.algal.2024.103538', True)
assert doi_matches('see (doi:10.1002/(SICI)1097-0258(19980430)17:8<873::AID-SIM779>3.0.CO;2-I)')[0][0].startswith('10.1002/(sici)')
assert doi_matches('DOI: 10.3390/w11122643, Received')[0] == ('10.3390/w11122643', True)
print('DOI patterns OK')

DOI patterns OK


## 3. Scoring: which DOI is the document's own?

In [6]:
def score_doi(cs):
    """cs = all places where ONE DOI was seen in ONE document. Returns (score, reasons)."""
    score, why = 0, []
    text_hits = [c for c in cs if c['source'] == 'text']
    edge_pages = {c['page'] for c in text_hits if c['zone'] in EDGE_ZONES}
    body_pages = {c['page'] for c in text_hits if c['zone'] == 'BODY'}

    if any(c['source'] == 'metadata' for c in cs):
        score += 6; why.append('in PDF metadata')
    if any(c['source'] == 'link' and c['page'] <= 2 for c in cs):
        score += 3; why.append('clickable link on page 1-2')
    if 1 in edge_pages:
        score += 4; why.append('header/footer/side on page 1')
    elif 2 in edge_pages:
        score += 2; why.append('header/footer/side on page 2')
    if len(edge_pages) >= 3:
        score += 4; why.append(f'repeated on {len(edge_pages)} page edges')
    elif len(edge_pages) == 2:
        score += 2; why.append('repeated on 2 page edges')
    if body_pages & {1, 2}:
        score += 2; why.append('body text on page 1-2')
    elif body_pages:
        score += 1; why.append('body text on page 3-6')
    if any(c['marked'] for c in text_hits):
        score += 2; why.append('labelled "doi"')
    only_refs = text_hits and all(c['zone'] == 'REFERENCES' for c in text_hits)
    if only_refs and not any(c['source'] in ('metadata', 'link') for c in cs):
        score -= 6; why.append('only in the reference list')
    return score, why

LOCATION_ORDER = ['METADATA', 'LINK', 'HEADER', 'FOOTER', 'SIDEBAR_LEFT', 'SIDEBAR_RIGHT',
                  'MARGIN_LEFT', 'MARGIN_RIGHT', 'BODY', 'REFERENCES']

def best_location(cs):
    best = min(cs, key=lambda c: (LOCATION_ORDER.index(c['zone']), c['page']))
    return best['zone'], best['page']

def choose_doi(cands):
    by_doi = defaultdict(list)
    for c in cands:
        by_doi[c['doi']].append(c)
    ranked = []
    for doi, cs in by_doi.items():
        score, why = score_doi(cs)
        zone, page = best_location(cs)
        ranked.append({'doi': doi, 'score': score, 'why': '; '.join(why), 'zone': zone, 'page': page})
    ranked.sort(key=lambda r: (-r['score'], r['page']))
    out = {'doi': '', 'doi_score': '', 'doi_evidence': '', 'doi_zone': '', 'doi_page': '',
           'doi_runner_up': '', 'doi_runner_up_score': '', 'doi_ambiguous': False, 'doi_candidates': len(ranked)}
    if ranked and ranked[0]['score'] >= MIN_SCORE:
        b = ranked[0]
        out.update(doi=b['doi'], doi_score=b['score'], doi_evidence=b['why'], doi_zone=b['zone'], doi_page=b['page'])
        out['doi_ambiguous'] = sum(r['score'] >= MIN_SCORE for r in ranked) > 1
    if len(ranked) > 1:
        out.update(doi_runner_up=ranked[1]['doi'], doi_runner_up_score=ranked[1]['score'])
    return out

## 4. Reading the PDFs

Every text line carries its position and direction. A line is assigned a **zone**: `HEADER` / `FOOTER` (top and bottom band), `MARGIN_LEFT` / `MARGIN_RIGHT` (horizontal text in the side bands), `SIDEBAR_LEFT` / `SIDEBAR_RIGHT` (vertical text near an edge, for example a rotated banner), or `BODY`. After a "References" heading, `BODY` lines become `REFERENCES`. Pages after `FULL_SCAN_PAGES` are only searched at their edges, because journals repeat their DOI in a running header or footer on every page.

In [7]:
def zone_of(bbox, direction, W, H):
    x0, y0, x1, y1 = bbox
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    if abs(direction[0]) < 0.7:                                   # vertical text
        if cx <= 2 * EDGE * W:
            return 'SIDEBAR_LEFT'
        if cx >= (1 - 2 * EDGE) * W:
            return 'SIDEBAR_RIGHT'
        return 'BODY'
    if cy <= EDGE * H:
        return 'HEADER'
    if cy >= (1 - EDGE) * H:
        return 'FOOTER'
    if x1 <= EDGE * W:
        return 'MARGIN_LEFT'
    if x0 >= (1 - EDGE) * W:
        return 'MARGIN_RIGHT'
    return 'BODY'

def candidate(document_id, doi, source, page, zone, marked):
    return {'document_id': document_id, 'doi': doi, 'source': source, 'page': page, 'zone': zone, 'marked': marked}

def read_pdf(pdf_path, document_id):
    """Returns (candidates, head_text, front_text). head = cover + page edges, front = pages 1-2."""
    cands, head, front = [], [], []
    with pymupdf.open(PDF_DIR / pdf_path) as pdf:
        info = ' '.join(str(v) for v in (pdf.metadata or {}).values() if v)
        xmp = pdf.get_xml_metadata() or ''
        for doi, marked in doi_matches(info + '\n' + xmp):
            cands.append(candidate(document_id, doi, 'metadata', 0, 'METADATA', marked))

        in_refs = False
        for number, page in enumerate(pdf, start=1):
            W, H = page.rect.width, page.rect.height
            if number <= LINK_PAGES:
                for link in page.get_links():
                    for doi, _ in doi_matches(unquote(link.get('uri') or '')):
                        cands.append(candidate(document_id, doi, 'link', number, 'LINK', True))

            ordered = []
            for block in page.get_text('dict', sort=True)['blocks']:
                if block.get('type') != 0:
                    continue
                for line in block['lines']:
                    text = clean(''.join(s['text'] for s in line['spans']))
                    if not text:
                        continue
                    zone = zone_of(line['bbox'], line['dir'], W, H)
                    if zone == 'BODY' and REF_HEADING.match(text):
                        in_refs = True
                    if zone == 'BODY' and in_refs:
                        zone = 'REFERENCES'
                    ordered.append((zone, text))

            groups = defaultdict(list)
            for zone, text in ordered:
                groups[zone].append(text)
            for zone, lines in groups.items():
                if number > FULL_SCAN_PAGES and zone not in EDGE_ZONES:
                    continue
                for doi, marked in doi_matches('\n'.join(lines)):
                    cands.append(candidate(document_id, doi, 'text', number, zone, marked))

            if number <= 2:
                page_text = '\n'.join(t for _, t in ordered)
                front.append(page_text)
                if number == 1:
                    head.append(page_text[:1500])
                head.append('\n'.join(t for z, t in ordered if z in EDGE_ZONES))
    return cands, '\n'.join(head), '\n'.join(front)

## 5. Document type from structure

Journal articles, conference papers, reports, theses and policy briefs are laid out differently, so the type is decided from **cues**: a cue found in the *head* of the document (cover, first 1,500 characters, header/footer lines of pages 1–2) counts 3 points, a cue found elsewhere on pages 1–2 counts 1 point. The type with the most points wins; ties follow the order thesis, policy brief, conference, journal, report. Below 2 points the type is `other`.

This is a first rule set: check it against your gold labels in notebook 09 and adjust the cues where it is wrong.

In [8]:
TYPE_CUES = {
    'thesis': [r'\bthesis\b', r'\bdissertation\b', r'\bproefschrift\b', r'\bscriptie\b',
               r'\bafstudeer\w*', r'\bin partial fulfil\w*'],
    'policy_brief': [r'\bpolicy brief\b', r'\bpolicy paper\b', r'\bbeleidsnotitie\b',
                     r'\bbeleidsbrief\b', r'\bfact ?sheet\b'],
    'conference_paper': [r'\bproceedings\b', r'\bconference\b', r'\bcongress\b', r'\bsymposium\b',
                         r'\bworkshop\b', r'\bconferentie\b', r'\bcongres\b'],
    'journal_article': [r'\bjournal\b', r'\btijdschrift\b', r'\bissn\b', r'\bvol(?:ume|\.)\s*\d',
                        r'\breceived\b[^\n]{0,80}\baccepted\b', r'\bpublished by\b',
                        r'\b(?:elsevier|springer|wiley|mdpi|sage|frontiers)\b',
                        r'\btaylor\s*&\s*francis\b', r'\bcorresponding author\b', r'\bopen access\b'],
    'report': [r'\brapport\w*', r'\breport\b', r'\beindrapportage\b', r'\bdeliverable\b',
               r'\bprojectplan\b', r'\bcolofon\b', r'\bcolophon\b', r'\bvoorwoord\b',
               r'\bopdrachtgever\b', r'\binhoudsopgave\b', r'\bmanagementsamenvatting\b'],
}
TYPE_ORDER = ['thesis', 'policy_brief', 'conference_paper', 'journal_article', 'report']

def classify_type(head, front):
    scores, cues = {}, {}
    for doc_type, patterns in TYPE_CUES.items():
        points, hits = 0, []
        for pattern in patterns:
            if re.search(pattern, head, re.I):
                points += 3; hits.append(pattern.replace(r'\b', ''))
            elif re.search(pattern, front, re.I):
                points += 1; hits.append(pattern.replace(r'\b', '') + ' (front)')
        scores[doc_type], cues[doc_type] = points, hits
    best = max(TYPE_ORDER, key=lambda t: (scores[t], -TYPE_ORDER.index(t)))
    if scores[best] < 2:
        return 'other', ''
    return best, '; '.join(cues[best])

assert classify_type('Journal of Water 2019, 11, 2643\nReceived: 1 May; Accepted: 2 June', '')[0] == 'journal_article'
assert classify_type('Proceedings of the 12th Symposium', '')[0] == 'conference_paper'
assert classify_type('Eindrapportage\nOpdrachtgever: Gemeente', '')[0] == 'report'
print('Type cues OK')

Type cues OK


## 6. Run on all documents

This is cheap (no model), so it runs on the whole collection. Candidates and decisions are saved for audit.

In [9]:
all_cands, rows, errors = [], [], []
for n, row in enumerate(todo.itertuples(index=False), start=1):
    try:
        cands, head, front = read_pdf(row.pdf_path, row.document_id)
    except Exception as exc:
        errors.append({'file_name': row.file_name, 'error': f'{type(exc).__name__}: {exc}'[:200]})
        continue
    all_cands += cands
    decision = choose_doi(cands)
    doc_type, cues = classify_type(head, front)
    rows.append({'document_id': row.document_id, 'file_name': row.file_name, **decision,
                 'document_type_rule': doc_type, 'type_cues': cues})
    if n % 20 == 0:
        print(f'{n}/{len(todo)} documents read')

features = pd.DataFrame(rows)
candidates_df = pd.DataFrame(all_cands, columns=['document_id', 'doi', 'source', 'page', 'zone', 'marked'])
features.to_csv(OUT_DIR / 'doc_features.csv', index=False, encoding='utf-8-sig')
candidates_df.to_csv(OUT_DIR / 'doi_candidates.csv', index=False, encoding='utf-8-sig')
print(f'{len(features)} documents, {len(candidates_df)} DOI sightings, {len(errors)} read errors')
if errors:
    display(pd.DataFrame(errors))

20/92 documents read
40/92 documents read
60/92 documents read
80/92 documents read
92 documents, 426 DOI sightings, 0 read errors


## 7. What was found, and where

In [10]:
found = features[features.doi != '']
print(f'DOI accepted for {len(found)} of {len(features)} documents ({len(found) / len(features):.0%}); '
      f'{int(features.doi_ambiguous.sum())} are ambiguous (more than one candidate above the threshold).\n')

print('Where the accepted DOI was seen (best location per document):')
display(found.groupby(['document_type_rule', 'doi_zone']).size().unstack(fill_value=0))

print('Document types guessed by the rules:')
display(features.document_type_rule.value_counts().to_frame('documents'))

# Compare with the simple DOI regex of notebook 01 (pages 1-3 text only)
obs = read_csv(INVENTORY_DIR / 'metadata_observations.csv')
old = obs[(obs.field_name == 'doi') & (obs.source_type == 'pdf_text')].groupby('document_id').observed_value.first().str.lower()
cmp = features.set_index('document_id')[['file_name', 'doi', 'doi_zone', 'doi_evidence']].join(old.rename('doi_notebook01'))
cmp['doi_notebook01'] = cmp.doi_notebook01.fillna('')
cmp['status'] = 'neither'
cmp.loc[(cmp.doi != '') & (cmp.doi_notebook01 == ''), 'status'] = 'only this notebook'
cmp.loc[(cmp.doi == '') & (cmp.doi_notebook01 != ''), 'status'] = 'only notebook 01'
cmp.loc[(cmp.doi != '') & (cmp.doi_notebook01 != ''), 'status'] = 'both agree'
cmp.loc[(cmp.doi != '') & (cmp.doi_notebook01 != '') & (cmp.doi != cmp.doi_notebook01), 'status'] = 'both, different DOI'
print('Agreement with the notebook 01 regex:')
display(cmp.status.value_counts().to_frame('documents'))
display(cmp[cmp.status.isin(['only this notebook', 'only notebook 01', 'both, different DOI'])]
        [['file_name', 'doi', 'doi_notebook01', 'doi_zone', 'doi_evidence', 'status']])

DOI accepted for 34 of 92 documents (37%); 22 are ambiguous (more than one candidate above the threshold).

Where the accepted DOI was seen (best location per document):


doi_zone,BODY,FOOTER,HEADER,LINK,METADATA,SIDEBAR_RIGHT
document_type_rule,,,,,,
journal_article,1,1,2,4,21,1
report,4,0,0,0,0,0


Document types guessed by the rules:


,documents
document_type_rule,
other,32
journal_article,30
report,27
conference_paper,2
thesis,1


Agreement with the notebook 01 regex:


,documents
status,
neither,58
both agree,32
only this notebook,1
"both, different DOI",1


,file_name,doi,doi_notebook01,doi_zone,doi_evidence,status
document_id,,,,,,
doc_05ac6631badec2a0,013_grip-op-teksten-grip-op-kennis.pdf,10.1080/0305569022000003681,,BODY,"body text on page 3-6; labelled ""doi""",only this notebook
doc_78a4ab0c8abaed20,028_towards-responsible-water-banking.pdf,10.1007/s13280-026-02459-y,10.1007/s13280-,METADATA,in PDF metadata; clickable link on page 1-2; h...,"both, different DOI"


## 8. Quick check against your gold file

Only the DOI and type columns, as counts. Notebook 09 has the full evaluation with confidence intervals, per-type breakdown and error diagnosis. An empty gold cell is skipped, `NONE` means no DOI exists.

In [11]:
GOLD_PATH = OUT_DIR / 'gold_metadata.csv'
if not GOLD_PATH.exists():
    print('No gold file yet.')
else:
    pred_cols = features[['document_id', 'doi', 'document_type_rule']].rename(columns={'doi': 'doi_pred'})
    gold = read_csv(GOLD_PATH).merge(pred_cols, on='document_id', how='left')
    norm = lambda s: re.sub(r'(?i)^https?://(dx\.)?doi\.org/', '', s.strip()).lower().rstrip('.')
    labelled = gold[gold.doi.str.strip() != '']
    truth = labelled.doi.map(lambda v: '' if v.strip().upper() == 'NONE' else norm(v))
    pred = labelled.doi_pred.fillna('').map(norm)
    outcome = pd.Series('TN (correctly empty)', index=labelled.index)
    outcome[(truth == '') & (pred != '')] = 'FP (spurious DOI)'
    outcome[(truth != '') & (pred == '')] = 'FN (missed)'
    outcome[(truth != '') & (pred != '') & (truth == pred)] = 'TP (correct)'
    outcome[(truth != '') & (pred != '') & (truth != pred)] = 'wrong DOI'
    print(f'DOI, {len(labelled)} labelled documents:')
    display(outcome.value_counts().to_frame('documents'))
    typed = gold[gold.document_type.str.strip() != '']
    if len(typed):
        print(f'Document type, {len(typed)} labelled documents: accuracy {(typed.document_type == typed.document_type_rule).mean():.2f}')
    else:
        print('No document types in the gold file yet; fill the column to check the type rules.')

DOI, 15 labelled documents:


,documents
TP (correct),15


No document types in the gold file yet; fill the column to check the type rules.
